In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/ml2022spring-hw1/covid.test.csv
/kaggle/input/competitions/ml2022spring-hw1/covid.train.csv


In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset,DataLoader

In [3]:
train_data=pd.read_csv("/kaggle/input/competitions/ml2022spring-hw1/covid.train.csv")
train_data.shape
x=train_data.iloc[:,1:117]
y=train_data.iloc[:,117:118]
x,y

(      AL  AK  AZ  AR  CA  CO  CT  FL  GA  ID  ...  travel_outside_state.4  \
 0      0   0   0   0   0   0   0   1   0   0  ...                7.944531   
 1      0   0   0   0   0   1   0   0   0   0  ...               10.523814   
 2      0   0   0   0   0   0   0   0   0   0  ...               16.477784   
 3      0   0   0   0   0   0   0   0   0   0  ...               19.140939   
 4      0   0   0   0   0   0   0   0   0   1  ...               22.698218   
 ...   ..  ..  ..  ..  ..  ..  ..  ..  ..  ..  ...                     ...   
 2694   0   0   0   0   0   0   0   0   0   0  ...               17.503116   
 2695   0   0   0   0   0   0   0   0   0   0  ...               13.804978   
 2696   0   0   0   1   0   0   0   0   0   0  ...               14.535434   
 2697   0   0   0   0   0   0   0   0   0   0  ...               10.993013   
 2698   0   0   0   0   0   0   0   0   0   0  ...               13.528217   
 
       work_outside_home.4     shop.4  restaurant.4  spent_tim

In [4]:
x_np=x.to_numpy()
y_np=y.to_numpy()
#在numpy下使用标准化一下，不然loss无法下降
x_min=x_np.min()
x_max=x_np.max()
x_np=(x_np-x_min)/(x_max-x_min)

#y_min=y_np.min()
#y_max=y_np.max()
#y_np=(y_np-y_min)/(y_max-y_min)

x_train=torch.tensor(x_np, dtype=torch.float32)
y_train=torch.tensor(y_np,dtype=torch.float32)


train_dataset=TensorDataset(x_train,y_train)
train_loader=DataLoader(train_dataset,batch_size=100,shuffle=True)

In [5]:

class LinearRegression(torch.nn.Module):
    def __init__(self):
        super(LinearRegression,self).__init__()

        self.layer=torch.nn.Linear(116,1)
    def forward(self,x):
        y_pred=self.layer(x)
        return y_pred

model=LinearRegression()
critrision=torch.nn.MSELoss(reduction="mean")
optimizer=torch.optim.SGD(model.parameters(),lr=0.1)

model.train()
for i in range(10001):
    for batch_x,batch_y in train_loader:
        y_pred=model(batch_x)
        loss=critrision(y_pred,batch_y)
        if(i%500==0):
            print(i,loss.item())
    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

0 103.86071014404297
0 92.78145599365234
0 140.87579345703125
0 132.04603576660156
0 110.7706069946289
0 92.01824188232422
0 86.1212387084961
0 72.98394012451172
0 63.17783737182617
0 67.23540496826172
0 57.215057373046875
0 47.71165084838867
0 44.699642181396484
0 44.0426025390625
0 32.83039093017578
0 30.56352996826172
0 24.1404972076416
0 20.954702377319336
0 23.57440948486328
0 26.11517906188965
0 24.612865447998047
0 23.394248962402344
0 16.35259437561035
0 18.055688858032227
0 17.10999298095703
0 15.980101585388184
0 24.208187103271484
500 2.657696485519409
500 3.3361802101135254
500 2.672822952270508
500 2.1949312686920166
500 2.10785174369812
500 2.16558837890625
500 1.9942551851272583
500 1.9894639253616333
500 1.5116145610809326
500 1.5818690061569214
500 2.0179102420806885
500 1.4379442930221558
500 1.406846046447754
500 1.6406419277191162
500 1.2162948846817017
500 1.063838243484497
500 1.5109120607376099
500 1.0425596237182617
500 1.2946239709854126
500 2.301976203918457
5

In [6]:
data_test=pd.read_csv("/kaggle/input/competitions/ml2022spring-hw1/covid.test.csv")
x_pd=data_test.iloc[:,1:117]
x_np=x_pd.to_numpy()
x_np=(x_np-x_min)/(x_max-x_min)

x_test=torch.tensor(x_np,dtype=torch.float32)           


In [7]:
test_ids = data_test.iloc[:, 0].values
model.eval() # 设置为评估模式，这是一个好习惯
with torch.no_grad(): # 关闭梯度计算，节省内存，加速推理
    y_pred_tensor = model(x_test)

# 将结果转为 numpy 数组，并压平成一维
# 模型输出是 [[1.2], [3.4]]，需要变成 [1.2, 3.4]
y_pred = y_pred_tensor.numpy().flatten()

# ==========================================
# 4. 生成并提交 CSV 文件
# ==========================================
# 创建一个 DataFrame，列名必须和 sample_submission.csv 中的一致
submission = pd.DataFrame({
    "id": test_ids,
    "tested_positive": y_pred 
})

# 保存为 csv 文件，index=False 表示不保存行号
submission.to_csv("submission.csv", index=False)

print("✅ 提交文件 submission.csv 已成功生成！")
print("文件前 5 行预览：")
print(submission.head())

✅ 提交文件 submission.csv 已成功生成！
文件前 5 行预览：
   id  tested_positive
0   0         8.546823
1   1         9.196710
2   2         4.924372
3   3         9.120351
4   4        16.615740
